# Populate sorting + decoding for all Berke lab sessions

Runs the **full sorting and decode pipeline** over every Berke lab session that needs it,
instead of one session at a time. This is the bulk version of
**`Berke_Lab_Sorting_and_Decode_V1.ipynb`** -- go there for the step-by-step walkthrough,
the plots, and the evaluation cells. To just *see* which sessions need what without running
anything, use **`Berke_Sorting_Decode_Candidates.ipynb`**.

Pipeline per session (exactly what the single-session notebook does):

1. `SpikeSortingRecordingSelection` -> `SpikeSortingRecording`
2. `ArtifactDetectionSelection` -> `ArtifactDetection`
3. `SpikeSortingSelection` -> `SpikeSorting`
4. `CurationV1` initial curation (`curation_id = 0`)
5. `MetricCurationSelection` -> `MetricCuration`
6. Round-2 curation at a **pinned** `curation_id` (1 = not-whitened, 3 = whitened)
7. Insert into the `SpikeSortingOutput` merge table
8. `SortedSpikesGroup` + `PositionGroup` -> `SortedSpikesDecodingSelection` -> `SortedSpikesDecodingV1`

**This is slow** -- sorting is hours per session. Every step is idempotent (selections use
`insert_selection` / `skip_duplicates`, populates skip what exists, and the round-2 curation
no-ops if its pinned `curation_id` already exists), so it's safe to interrupt and re-run.

## Parameters

Same parameters as the single-session notebook.

In [ ]:
import datajoint as dj
import numpy as np
import pandas as pd

import spyglass.common as sgc
import spyglass.spikesorting.v1 as sgs
import spyglass.position as sgp
from spyglass.common import Raw, AnalysisNwbfile
from spyglass.position import PositionOutput
from spyglass.decoding.decoding_merge import DecodingOutput
from spyglass.spikesorting.spikesorting_merge import SpikeSortingOutput
from spyglass.spikesorting.analysis.v1.group import SortedSpikesGroup
from spyglass.decoding.v1.core import PositionGroup
from spyglass.decoding import SortedSpikesDecodingSelection
from spyglass.decoding.v1.sorted_spikes import SortedSpikesDecodingV1
from spyglass_hexmaze.hex_maze_behavior import HexMazeBlock

# Preprocessing
preproc_param_name = "default"
artifact_param_name = "ampl_1000_z_30_prop_075_1ms"
TEAM_NAME = "Berke lab and friends"
# Sorting
sorter = "mountainsort4"
sorter_param_name = "franklab_probe_v1_30KHz"
# Curation
waveform_param_name = "default_not_whitened"
metric_param_name = "franklab_default"
metric_curation_param_name = "default"
# Decoding
unit_filter_params_name = "default_exclusion"
decoding_param_name = "contfrag_sorted_50chunks_100blocks"
pos_group_name = "sorted_spikes_pos_group"
trodes_pos_params_name = "berke_double_led_decoding"

# Each waveform param gets its own sorted spikes group (and therefore its own decode)
if waveform_param_name == "default_whitened":
    ss_group_name = "sorted_spikes_group_default_whitened"
else:
    ss_group_name = "sorted_spikes_group"

# Round-2 curation writes to a FIXED curation_id per waveform branch, so the id alone
# identifies the branch and re-runs can't create duplicates. 2 is skipped so the two
# branches can never collide.
CURATION_ID_BY_WAVEFORM = {"default_not_whitened": 1, "default_whitened": 3}
target_curation_id = CURATION_ID_BY_WAVEFORM[waveform_param_name]
target_description = f"after metric curation ({waveform_param_name})"

print(f"waveform param '{waveform_param_name}' -> curation_id {target_curation_id}, group '{ss_group_name}'")

### GPU (for decoding)

Optional -- set the device before decoding, same as the single-session notebook.

In [ ]:
# import jax
# device_id = 6  # or whichever GPU you want
# jax.config.update("jax_default_device", jax.devices()[device_id])

## Find the Berke sessions that need work

Same detection as `Berke_Sorting_Decode_Candidates.ipynb`: a session is runnable when it has
raw ephys and sort groups. Sessions with no ephys (behavior / photometry only) or no sort
groups are skipped -- create sort groups first for those.

In [ ]:
hex_sessions = sorted(set(HexMazeBlock.fetch('nwb_file_name')))
session_keys = [{'nwb_file_name': s} for s in hex_sessions]
session_lab = pd.DataFrame(
    (sgc.Session & session_keys).fetch('nwb_file_name', 'subject_id', 'lab_name', as_dict=True)
)
berke_sessions = sorted(session_lab.loc[session_lab['lab_name'] == 'Berke Lab', 'nwb_file_name'])
berke_keys = [{'nwb_file_name': s} for s in berke_sessions]

ephys_sessions = set((Raw & berke_keys).fetch('nwb_file_name'))
sort_group_sessions = {s for s in berke_sessions if len(sgs.SortGroup & {'nwb_file_name': s}) > 0}
sorted_sessions = {
    s for s in berke_sessions
    if len(SpikeSortingOutput().get_restricted_merge_ids(
        {'nwb_file_name': s}, sources=['v0', 'v1'], restrict_by_artifact=False))
}

# Runnable = has the raw data and the sort groups the pipeline needs
runnable = [s for s in berke_sessions if s in ephys_sessions and s in sort_group_sessions]
needs_sorting = [s for s in runnable if s not in sorted_sessions]

print(f"{len(berke_sessions)} Berke sessions: {len(ephys_sessions)} with ephys, "
      f"{len(sort_group_sessions)} with sort groups")
print(f"\n{len(runnable)} runnable, of which {len(needs_sorting)} still need sorting:")
for s in needs_sorting:
    print(f"    {s}")
print(f"\n(sessions with ephys but NO sort groups -- create sort groups first: "
      f"{sorted(set(ephys_sessions) - sort_group_sessions)})")

## Pipeline helpers

`insert_metric_curation_at` is the pinned-`curation_id` version of
`CurationV1.insert_metric_curation` (stock auto-assigns `max+1`, which drifts on re-runs and
creates duplicates). Copied from the single-session notebook.

In [ ]:
from spyglass.spikesorting.v1.curation import _write_sorting_to_nwb_with_curation


def insert_metric_curation_at(mc_key, curation_id, description, apply_merge=False):
    """Like CurationV1.insert_metric_curation, but writes to a SPECIFIC curation_id
    instead of auto-assigning max+1. Idempotent: if that curation_id already exists
    for this sorting, nothing is written and the existing key is returned."""
    sorting_id, parent_curation_id = (
        sgs.MetricCurationSelection & mc_key
    ).fetch1("sorting_id", "curation_id")
    sorting_id = str(sorting_id)

    # Idempotency: if this branch's curation_id already exists, leave it alone.
    target_key = {"sorting_id": sorting_id, "curation_id": curation_id}
    if len(sgs.CurationV1 & target_key):
        return target_key

    labels = sgs.MetricCuration.get_labels(mc_key) or None
    merge_groups = sgs.MetricCuration.get_merge_groups(mc_key) or None
    analysis_file_name, object_id = _write_sorting_to_nwb_with_curation(
        sorting_id=sorting_id,
        labels=labels,
        merge_groups=merge_groups,
        metrics=None,
        apply_merge=apply_merge,
    )
    AnalysisNwbfile().add(
        (sgs.SpikeSortingSelection & {"sorting_id": sorting_id}).fetch1("nwb_file_name"),
        analysis_file_name,
    )
    sgs.CurationV1.insert1(
        {
            "sorting_id": sorting_id,
            "curation_id": curation_id,
            "parent_curation_id": parent_curation_id,
            "analysis_file_name": analysis_file_name,
            "object_id": object_id,
            "merges_applied": apply_merge,
            "description": description,
        },
        skip_duplicates=True,
    )
    return target_key


def get_run_epoch_and_interval(nwb_file_name):
    """The (epoch, run interval) to sort/decode. Berke sessions have one run epoch (00_r1)."""
    epochs = sorted(set((HexMazeBlock & {"nwb_file_name": nwb_file_name}).fetch("epoch")))
    epoch = epochs[0]
    interval = (sgc.TaskEpoch & {"nwb_file_name": nwb_file_name, "epoch": epoch}).fetch1(
        "interval_list_name"
    )
    return epoch, interval

In [ ]:
def run_sorting_for_session(nwb_file_name):
    """Steps 1-7: raw recording through to the SpikeSortingOutput merge table."""
    epoch, run_interval = get_run_epoch_and_interval(nwb_file_name)
    print(f"  epoch {epoch}, interval {run_interval!r}")

    sort_group_ids = (sgs.SortGroup & {"nwb_file_name": nwb_file_name}).fetch("sort_group_id")
    print(f"  {len(sort_group_ids)} sort group(s)")

    # 1. Recording selection + populate
    group_keys = []
    for sort_group_id in sort_group_ids:
        key = {
            "nwb_file_name": nwb_file_name,
            "sort_group_id": sort_group_id,
            "interval_list_name": run_interval,
            "preproc_param_name": preproc_param_name,
            "team_name": TEAM_NAME,
        }
        sgs.SpikeSortingRecordingSelection.insert_selection(key)
        group_keys.append((sgs.SpikeSortingRecordingSelection & key).fetch1("KEY"))
    print("  populating SpikeSortingRecording...")
    sgs.SpikeSortingRecording.populate(group_keys)

    # 2. Artifact detection
    artifact_keys = []
    for gk in group_keys:
        key = {"recording_id": gk["recording_id"], "artifact_param_name": artifact_param_name}
        sgs.ArtifactDetectionSelection.insert_selection(key)
        artifact_keys.append((sgs.ArtifactDetectionSelection & key).fetch1("KEY"))
    print("  populating ArtifactDetection...")
    sgs.ArtifactDetection.populate(artifact_keys)

    # 3. Spike sorting. Artifact detection doesn't always produce an interval list, so we
    # check it exists and move forward with the sort groups that did.
    spike_sorting_keys = []
    for gk in group_keys:
        art_id = (
            sgs.ArtifactDetectionSelection & {"recording_id": gk["recording_id"]}
        ).fetch1("artifact_id")
        if len(sgc.IntervalList() & {"interval_list_name": str(art_id)}) == 0:
            print(f"    no interval list for artifact {art_id}, skipping this sort group")
            continue
        ss_key = {
            "recording_id": gk["recording_id"],
            "sorter": sorter,
            "nwb_file_name": nwb_file_name,
            "interval_list_name": str(art_id),
            "sorter_param_name": sorter_param_name,
        }
        sgs.SpikeSortingSelection.insert_selection(ss_key)
        spike_sorting_keys.append((sgs.SpikeSortingSelection & ss_key).proj().fetch1("KEY"))
    print(f"  populating SpikeSorting ({len(spike_sorting_keys)} sorting(s))... this is the slow part")
    sgs.SpikeSorting.populate(spike_sorting_keys)

    # 4. Initial automatic curation (curation_id = 0)
    for ss_key in spike_sorting_keys:
        initial = {"sorting_id": str(ss_key["sorting_id"]), "curation_id": 0}
        if len(sgs.CurationV1 & initial) == 0:
            sgs.CurationV1.insert_curation(
                sorting_id=str(ss_key["sorting_id"]),
                description="initial automatic curation",
            )

    # 5. Metric curation
    metric_curation_keys = []
    for ss_key in spike_sorting_keys:
        mc_key = {
            "sorting_id": str(ss_key["sorting_id"]),
            "curation_id": 0,
            "waveform_param_name": waveform_param_name,
            "metric_param_name": metric_param_name,
            "metric_curation_param_name": metric_curation_param_name,
        }
        sgs.MetricCurationSelection.insert_selection(mc_key)
        metric_curation_keys.append((sgs.MetricCurationSelection & mc_key).fetch1("KEY"))
    print("  populating MetricCuration...")
    sgs.MetricCuration.populate(metric_curation_keys)

    # 6. Round-2 curation at the pinned curation_id for this waveform branch
    curation_keys_round2 = [
        insert_metric_curation_at(mc_key, target_curation_id, target_description)
        for mc_key in metric_curation_keys
    ]

    # 7. Into the merge table for downstream use
    for key in curation_keys_round2:
        merge_insert_key = (sgs.CurationV1 & key).fetch("KEY", as_dict=True)
        SpikeSortingOutput.insert(merge_insert_key, part_name="CurationV1", skip_duplicates=True)

    print(f"  sorting done: {len(curation_keys_round2)} curation(s) in SpikeSortingOutput")
    return curation_keys_round2

In [ ]:
def run_decode_for_session(nwb_file_name):
    """Step 8: SortedSpikesGroup + PositionGroup -> SortedSpikesDecodingV1."""
    epoch, run_interval = get_run_epoch_and_interval(nwb_file_name)

    # Only this run's waveform branch (curation_id), so whitened / not-whitened don't mix
    sorter_keys = {
        "nwb_file_name": nwb_file_name,
        "sorter": sorter,
        "curation_id": target_curation_id,
    }
    spikesorting_merge_ids = SpikeSortingOutput().get_restricted_merge_ids(
        sorter_keys, restrict_by_artifact=False
    )
    if len(spikesorting_merge_ids) == 0:
        raise RuntimeError(f"no sorted units for curation_id {target_curation_id}")
    print(f"  {len(spikesorting_merge_ids)} spikesorting merge id(s)")

    # SortedSpikesGroup
    if len(SortedSpikesGroup & {"nwb_file_name": nwb_file_name,
                                "sorted_spikes_group_name": ss_group_name}) == 0:
        SortedSpikesGroup().create_group(
            group_name=ss_group_name,
            nwb_file_name=nwb_file_name,
            keys=[{"spikesorting_merge_id": mid} for mid in spikesorting_merge_ids],
            unit_filter_params_name=unit_filter_params_name,
        )

    # Position with the decoding params (PositionGroup is built from TrodesPosV1)
    position_selection_key = {
        "nwb_file_name": nwb_file_name,
        "interval_list_name": f"pos {epoch} valid times",
        "trodes_pos_params_name": trodes_pos_params_name,
    }
    sgp.v1.TrodesPosSelection.insert1(position_selection_key, skip_duplicates=True)
    position_key = (sgp.v1.TrodesPosSelection() & position_selection_key).fetch1("KEY")
    if len(PositionOutput.TrodesPosV1() & position_key) == 0:
        print("  populating TrodesPosV1...")
        sgp.v1.TrodesPosV1.populate(position_key)

    # PositionGroup
    position_merge_ids = (PositionOutput.TrodesPosV1 & position_key).fetch("merge_id")
    if len(PositionGroup & {"nwb_file_name": nwb_file_name,
                            "position_group_name": pos_group_name}) == 0:
        PositionGroup().create_group(
            nwb_file_name=nwb_file_name,
            group_name=pos_group_name,
            keys=[{"pos_merge_id": mid} for mid in position_merge_ids],
            upsample_rate=500,
        )

    # Decode over the whole run epoch
    selection_key = {
        "sorted_spikes_group_name": ss_group_name,
        "unit_filter_params_name": unit_filter_params_name,
        "position_group_name": pos_group_name,
        "decoding_param_name": decoding_param_name,
        "nwb_file_name": nwb_file_name,
        "encoding_interval": run_interval,
        "decoding_interval": run_interval,
        "estimate_decoding_params": True,
    }
    SortedSpikesDecodingSelection.insert1(selection_key, skip_duplicates=True)
    if len(SortedSpikesDecodingV1 & selection_key) == 0:
        print("  populating SortedSpikesDecodingV1...")
        SortedSpikesDecodingV1.populate(selection_key)
    else:
        print("  decode already exists")
    return selection_key

## Run it

**Start with one session** to confirm the parameters and environment are right, then raise
`LIMIT`. Each session is hours of sorting. Errors are caught per session so one bad session
doesn't kill the batch -- the failures are collected and printed at the end.

In [ ]:
# Which sessions to run. Defaults to everything detected as needing sorting.
SESSIONS_TO_RUN = needs_sorting

# Safety: start with 1 to check params/environment end to end, then raise (None = all)
LIMIT = 1
todo = SESSIONS_TO_RUN[:LIMIT] if LIMIT else SESSIONS_TO_RUN

print(f"Running {len(todo)} of {len(SESSIONS_TO_RUN)} session(s): {todo}\n")

failures = {}
for i, nwb_file_name in enumerate(todo, 1):
    print(f"[{i}/{len(todo)}] {nwb_file_name}")
    try:
        run_sorting_for_session(nwb_file_name)
        run_decode_for_session(nwb_file_name)
        print(f"  DONE {nwb_file_name}\n")
    except Exception as e:
        failures[nwb_file_name] = repr(e)
        print(f"  FAILED {nwb_file_name}: {e!r}\n")

print(f"Finished. {len(todo) - len(failures)} succeeded, {len(failures)} failed.")
for nwb, err in failures.items():
    print(f"  {nwb}: {err}")

## Check the results

In [ ]:
# What ended up in the merge tables for the sessions we just ran
for nwb_file_name in todo:
    n_sorted = len(SpikeSortingOutput().get_restricted_merge_ids(
        {'nwb_file_name': nwb_file_name}, sources=['v1'], restrict_by_artifact=False))
    n_decoded = len(DecodingOutput.SortedSpikesDecodingV1 & {'nwb_file_name': nwb_file_name})
    print(f"{nwb_file_name}: {n_sorted} sorting merge id(s), {n_decoded} decode(s)")